# tsfm-peft on Colab

Runs every benchmark arm on a Colab GPU and produces the metrics artifacts the README
results table is generated from.

**This notebook computes nothing itself.** Every number comes from `tsfm-peft run`, the same
entry point a local run uses, against the configs in the repository. The notebook clones,
installs, orders the runs, resumes after a disconnect, and packages the artifacts — that is
all. If it contained its own training loop the published numbers would no longer come from
the configs sitting next to them in the repo, and the table would stop being checkable.

**What you need:** a Colab runtime with a GPU (Runtime, Change runtime type, T4 is enough)
and roughly 1.5 GB of Google Drive if you want the run to survive a disconnect.

**How to use it:** run the cells top to bottom. If the session dies, reconnect and run them
top to bottom again — completed arms are skipped, so you pick up where you left off.

## 1. Check the GPU

In [ ]:
# The arms differ in wall-clock and peak memory, so every row has to come from the same
# hardware for those columns to mean anything. This prints what you got; if a later session
# lands on a different GPU, cell 6 will say so before you waste a run on it.
import subprocess


def nvidia_smi(query):
    """Query nvidia-smi, returning "" when there is no GPU (and so no nvidia-smi at all)."""
    try:
        out = subprocess.run(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
            capture_output=True,
            text=True,
        )
    except (OSError, subprocess.SubprocessError):
        return ""
    return out.stdout.strip() if out.returncode == 0 else ""


print(
    nvidia_smi("name,memory.total,driver_version")
    or "NO GPU. Runtime > Change runtime type > T4 GPU, then rerun this cell."
)

## 2. Settings

`REF` pins what you are benchmarking. Leave it on a branch to track the latest, or set it to
a commit sha to reproduce a specific published table — the sha ends up in every artifact, so
the README can say which commit produced its numbers.

In [ ]:
REPO_URL = "https://github.com/abhijeetkumar1/tsfm-peft.git"
REF = "dev"  # branch, tag, or commit sha

USE_DRIVE = True  # False keeps everything in the ephemeral runtime and loses it on a disconnect
DRIVE_DIR = "MyDrive/tsfm-peft"

REPO = "/content/tsfm-peft"

## 3. Mount Drive

The dataset files and the 231M-parameter checkpoint are downloaded once and reused. Putting
them on Drive alongside the artifacts is what makes a disconnect cost you only the arm that
was running, rather than the whole session.

In [ ]:
import os
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    base = Path("/content/drive") / DRIVE_DIR
else:
    base = Path("/content/tsfm-peft-run")

CACHE = base / "cache"
RESULTS = base / "results"
for directory in (CACHE, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)

# The package reads both of these; nothing is written inside the repository.
os.environ["TSFM_PEFT_CACHE"] = str(CACHE)
os.environ["TSFM_PEFT_RESULTS"] = str(RESULTS)
os.environ["HF_HOME"] = str(CACHE / "huggingface")
os.environ["PYTHONHASHSEED"] = "0"

print("cache   ", CACHE)
print("results ", RESULTS)

## 4. Clone and install

Installing *inside the clone* matters: the package records `git rev-parse HEAD` from the
working directory into every artifact, which is what lets the README name the commit its
numbers came from. Install from a bare wheel instead and that field is empty.

Colab's preinstalled CUDA torch already satisfies the requirement, so it is left alone.
`transformers` is upgraded, which normally means restarting the runtime — but every
experiment below runs as a subprocess, so this kernel's stale copy never gets used and no
restart is needed.

In [ ]:
import shutil
import subprocess
from pathlib import Path

if Path(REPO).exists():
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True
).stdout.strip()
print("commit", commit)

subprocess.run(["pip", "install", "--quiet", "-e", ".[models]"], cwd=REPO, check=True)
print(
    subprocess.run(
        ["tsfm-peft", "--version"], cwd=REPO, capture_output=True, text=True
    ).stdout.strip()
)

## 5. The arms

Ordered so that a session cut short still leaves you something publishable. The baselines and
zero-shot rows come first (they are the reference every fine-tuned arm has to beat), then the
headline LoRA and DoRA arms on both datasets, and only then the rank ablation.

The list is checked against the configs actually in the repo, so an arm added later cannot be
silently skipped here.

In [ ]:
from pathlib import Path

ARMS = [
    # Seconds each. No weights, no GPU needed.
    "etth1-seasonal-naive",
    "nn5_daily-seasonal-naive",
    # Minutes each. The reference rows.
    "etth1-timesfm-zeroshot",
    "nn5_daily-timesfm-zeroshot",
    # The headline comparison, both datasets.
    "etth1-timesfm-lora",
    "etth1-timesfm-dora",
    "nn5_daily-timesfm-lora",
    "nn5_daily-timesfm-dora",
    # The rank ablation. Nice to have; the table above is complete without it.
    "etth1-timesfm-lora-r4",
    "etth1-timesfm-lora-r8",
    "etth1-timesfm-lora-r32",
]

CONFIGS = Path(REPO) / "configs" / "experiments"
# Synthetic arms are CI smoke fixtures against a randomly initialised checkpoint. They are
# excluded from the results table by construction, so running them here would prove nothing.
on_disk = {p.stem for p in CONFIGS.glob("*.yaml") if not p.stem.startswith("synthetic-")}

missing = on_disk - set(ARMS)
unknown = set(ARMS) - on_disk
if missing or unknown:
    raise SystemExit(
        f"ARMS is out of step with {CONFIGS}: missing {sorted(missing)}, unknown {sorted(unknown)}"
    )
print(f"{len(ARMS)} arms to run")

## 6. Run

Each arm writes one JSON artifact named after itself, so an arm that already has one is
skipped. Rerun this cell as many times as it takes; it only ever does the work still
outstanding.

A failing arm does not stop the others — it is reported at the end and you can rerun to retry
it.

In [ ]:
import json
import subprocess
import time
from pathlib import Path


def gpu_name():
    """The GPU this session has, as the artifacts record it. "CPU" when there is none."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True,
            text=True,
        )
    except (OSError, subprocess.SubprocessError):
        return "CPU"
    name = out.stdout.strip() if out.returncode == 0 else ""
    return name.splitlines()[0] if name else "CPU"


def recorded_gpu(path):
    """The GPU an existing artifact was produced on, or None."""
    devices = json.loads(Path(path).read_text())["environment"]["hardware"]["devices"]
    return devices[0]["name"] if devices else "CPU"


current = gpu_name()
done = sorted(Path(RESULTS).glob("*.json"))
if done:
    previous = {recorded_gpu(p) for p in done}
    if previous != {current}:
        print(
            f"WARNING: artifacts on disk were produced on {sorted(previous)} but this "
            f"session has {current}.\n"
            "         Accuracy stays valid, but the peak-memory and train-time columns "
            "will not be comparable\n"
            "         across rows, and the generated table will say so. Delete the old "
            "artifacts to redo them here.\n"
        )

failed = []
for i, arm in enumerate(ARMS, 1):
    artifact = Path(RESULTS) / f"{arm}.json"
    if artifact.exists():
        print(f"[{i}/{len(ARMS)}] {arm}: already done, skipping")
        continue

    print(f"[{i}/{len(ARMS)}] {arm}: running", flush=True)
    started = time.monotonic()
    result = subprocess.run(["tsfm-peft", "run", str(CONFIGS / f"{arm}.yaml")], cwd=REPO)
    elapsed = time.monotonic() - started

    if result.returncode == 0:
        print(f"[{i}/{len(ARMS)}] {arm}: done in {elapsed / 60:.1f} min\n", flush=True)
    else:
        failed.append(arm)
        print(
            f"[{i}/{len(ARMS)}] {arm}: FAILED after {elapsed / 60:.1f} min "
            f"(exit {result.returncode})\n",
            flush=True,
        )

remaining = [a for a in ARMS if not (Path(RESULTS) / f"{a}.json").exists()]
print(f"\ncomplete: {len(ARMS) - len(remaining)}/{len(ARMS)}")
if failed:
    print("failed this pass:", ", ".join(failed))
if remaining:
    print("still to run:", ", ".join(remaining), "-- rerun this cell")

## 7. Preview the table

The same generator the README uses, run against what you have so far. Arms still outstanding
show as `--` rather than as a blank, so a partial table cannot be mistaken for a finished one.

In [ ]:
import subprocess

print(
    subprocess.run(
        ["tsfm-peft", "table", "--results-dir", str(RESULTS), "--configs", str(CONFIGS)],
        cwd=REPO,
        capture_output=True,
        text=True,
    ).stdout
)

## 8. Download the artifacts

Unzip into `results/` at the root of your local clone, then:

```bash
uv run tsfm-peft table --configs configs/experiments --write
```

That rewrites the README table from the artifacts and is the only way a number gets in.
Commit the artifacts alongside the README change so every published figure has its
provenance — config, seed, commit, package versions and hardware — in the repository.

Deliberately no `git push` from here: it would mean putting a GitHub token in a notebook.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/tsfm-peft-results", "zip", RESULTS)
print(archive)
files.download(archive)